#WiDS Hackathon: ADHD and Sex Prediciton Based on Numerical, Categorical and MRI Data

Importing Libraries

In [38]:
# Data manipulation and analysis
import numpy as np
import pandas as pd

# Scikit-learn preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras for deep learning
import tensorflow as tf
from tensorflow import keras
from keras import layers, Model, Input, regularizers
from keras.callbacks import EarlyStopping

Importing Training and Test Data

In [39]:
train_path = "/content/drive/MyDrive/wids2025"

cat_df = pd.read_excel(f"{train_path}/TRAIN_CATEGORICAL_METADATA_new.xlsx")
mri_df = pd.read_csv(f"{train_path}/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
quant_df = pd.read_excel(f"{train_path}/TRAIN_QUANTITATIVE_METADATA_new.xlsx")
label_df = pd.read_excel(f"{train_path}/TRAINING_SOLUTIONS.xlsx")

test_cat_df = pd.read_excel(f"{train_path}/TEST_CATEGORICAL.xlsx")
test_mri_df = pd.read_csv(f"{train_path}/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")
test_quant_df = pd.read_excel(f"{train_path}/TEST_QUANTITATIVE_METADATA.xlsx")

sample_submission = pd.read_excel(f"{train_path}/SAMPLE_SUBMISSION.xlsx")

In [40]:
print("Categorical metadata shape :", cat_df.shape)
print("Quantitative metadata shape:", quant_df.shape)
print("Labels shape               :", label_df.shape)
print("MRI data shape             :", mri_df.shape)

Categorical metadata shape : (1213, 10)
Quantitative metadata shape: (1213, 19)
Labels shape               : (1213, 3)
MRI data shape             : (1213, 19901)


Cleaning Training Data

In [41]:
train_df = cat_df.merge(quant_df, on="participant_id")
train_df = train_df.merge(mri_df, on="participant_id")
train_df = train_df.merge(label_df, on="participant_id")

meta_categorical_cols = list(cat_df.columns.drop("participant_id"))
meta_numerical_cols   = list(quant_df.columns.drop("participant_id"))
mri_columns           = list(mri_df.columns.drop("participant_id"))

meta_columns = meta_categorical_cols + meta_numerical_cols

X_meta_raw = train_df[meta_columns]
X_mri      = train_df[mri_columns].values
y_adhd     = train_df['ADHD_Outcome'].values
y_sex      = train_df['Sex_F'].values

X_cat = X_meta_raw[meta_categorical_cols]
X_num = X_meta_raw[meta_numerical_cols]

print("NaNs in categorical features:", X_cat.isnull().sum().sum())
print("NaNs in numerical features  :", X_num.isnull().sum().sum())

X_cat_clean = X_cat.fillna(X_cat.mode().iloc[0])
X_num_clean = X_num.fillna(X_num.mean())

X_meta_clean = pd.concat([X_cat_clean, X_num_clean], axis=1)

print("NaNs in cleaned categorical features:", X_cat_clean.isnull().sum().sum())
print("NaNs in cleaned numerical features  :", X_num_clean.isnull().sum().sum())

NaNs in categorical features: 566
NaNs in numerical features  : 549
NaNs in cleaned categorical features: 0
NaNs in cleaned numerical features  : 0


Processing Training Data
1. Categorical Data: One Hot Encoding
2. MRI Data: Scaling and PCA

In [42]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat_encoded = ohe.fit_transform(X_cat_clean)

X_num_array = X_num_clean.values

X_meta = np.hstack([X_cat_encoded, X_num_array])

scaler_mri = StandardScaler()
X_mri_scaled = scaler_mri.fit_transform(X_mri)

pca = PCA(n_components=0.95, random_state=42)
X_mri_pca = pca.fit_transform(X_mri_scaled)

print(f"X_mri_pca shape: {X_mri_pca.shape}")
print(f"PCA retained {np.sum(pca.explained_variance_ratio_)*100:.2f}% variance.")

X_mri_pca shape: (1213, 909)
PCA retained 95.01% variance.


## Neural Network Model

Test Train Split 80-20

In [43]:
X_meta_train, X_meta_val, X_mri_train, X_mri_val, y_adhd_train, y_adhd_val, y_sex_train, y_sex_val = train_test_split(
    X_meta, X_mri_pca, y_adhd, y_sex,
    test_size=0.2,
    random_state=42,
    stratify=y_adhd
)

Building Neural Network Model

Hyperparameter Summary

| Hyperparameter | Value(s) | Location |
|---|---|---|
| L2 Regularization | `1e-4` | Dense layers (both branches, merged) |
| Neurons per Dense Layer | Metadata: 64, 32; MRI: 128, 64; Merged: 64 | Respective layers |
| Activation Function | ReLU (hidden), Sigmoid (output) | Respective layers |
| Dropout Rate | 0.5 | Both branches, merged layer |
| Batch Normalization | Applied | After first Dense layer in branches |
| Input Shape | Metadata: Dynamic; MRI: `(909,)` | Input layers |
| Optimizer | `adam` | Model compilation |
| Loss Function | ADHD: weighted; Sex: binary cross-entropy | Model compilation |
| Metrics | Accuracy | Model compilation |
| Early Stopping | `patience=3`, monitor `val_loss` | Training process |

In [50]:
l2_reg = regularizers.l2(1e-4)

meta_input = Input(shape=(X_meta.shape[1],), name="meta_input")
x_meta = layers.Dense(64, activation="relu", kernel_regularizer=l2_reg)(meta_input)
x_meta = layers.BatchNormalization()(x_meta)
x_meta = layers.Dropout(0.5)(x_meta)
x_meta = layers.Dense(32, activation="relu", kernel_regularizer=l2_reg)(x_meta)

mri_input = Input(shape=(909,), name="mri_input")
x_mri = layers.Dense(128, activation="relu", kernel_regularizer=l2_reg)(mri_input)
x_mri = layers.BatchNormalization()(x_mri)
x_mri = layers.Dropout(0.5)(x_mri)
x_mri = layers.Dense(64, activation="relu", kernel_regularizer=l2_reg)(x_mri)

merged = layers.concatenate([x_meta, x_mri])
merged = layers.Dense(64, activation="relu", kernel_regularizer=l2_reg)(merged)
merged = layers.Dropout(0.5)(merged)

adhd_output = layers.Dense(1, activation="sigmoid", name="adhd_output")(merged)
sex_output = layers.Dense(1, activation="sigmoid", name="sex_output")(merged)

model = Model(inputs=[meta_input, mri_input], outputs=[adhd_output, sex_output])

In [51]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

class_weights_adhd = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_adhd_train),
    y=y_adhd_train
)

class_weights_adhd = dict(zip(np.unique(y_adhd_train), class_weights_adhd))

print("Class Weights for ADHD Outcome:", class_weights_adhd)

def weighted_binary_crossentropy(y_true, y_pred):
  weights = tf.where(tf.equal(y_true, 1),
                    tf.cast(class_weights_adhd[1], tf.float32),
                    tf.cast(class_weights_adhd[0], tf.float32))
  return tf.keras.losses.binary_crossentropy(y_true, y_pred) * weights

Class Weights for ADHD Outcome: {np.int64(0): np.float64(1.5901639344262295), np.int64(1): np.float64(0.7293233082706767)}


Compiling Model

In [52]:
model.compile(
    optimizer="adam",
    loss={
        "adhd_output": weighted_binary_crossentropy,
        "sex_output": "binary_crossentropy"
    },
    metrics={"adhd_output": "accuracy", "sex_output": "accuracy"}
)

Training Model

In [54]:
history = model.fit(
    [X_meta_train, X_mri_train],
    [y_adhd_train, y_sex_train],
    epochs=50,
    batch_size=32,
    validation_data=([X_meta_val, X_mri_val], [y_adhd_val, y_sex_val]),
    callbacks=[early_stopping]
)

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 135ms/step - adhd_output_accuracy: 0.5867 - adhd_output_loss: 0.8547 - loss: 1.7094 - sex_output_accuracy: 0.5637 - sex_output_loss: 0.8048 - val_adhd_output_accuracy: 0.5926 - val_adhd_output_loss: 0.7005 - val_loss: 1.3988 - val_sex_output_accuracy: 0.6667 - val_sex_output_loss: 0.6491
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - adhd_output_accuracy: 0.6182 - adhd_output_loss: 0.7347 - loss: 1.4745 - sex_output_accuracy: 0.6167 - sex_output_loss: 0.6900 - val_adhd_output_accuracy: 0.6667 - val_adhd_output_loss: 0.6498 - val_loss: 1.3319 - val_sex_output_accuracy: 0.6543 - val_sex_output_loss: 0.6308
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - adhd_output_accuracy: 0.7132 - adhd_output_loss: 0.5696 - loss: 1.2710 - sex_output_accuracy: 0.6359 - sex_output_loss: 0.6516 - val_adhd_output_accuracy: 0.6955 - val_adhd_output_loss: 0.6231 - val_loss: 1.2856 - val_sex_output_accuracy: 0.6708 - val_sex_output_loss: 0.6119
Epoch 4/50
31/

Preparing Test Data For Prediciton

In [55]:
categorical_cols = list(test_cat_df.columns.drop('participant_id'))
quantitative_cols = list(test_quant_df.columns.drop('participant_id'))

test_cat_df[meta_categorical_cols] = test_cat_df[meta_categorical_cols].apply(lambda x: x.fillna(x.mode()[0]))
X_cat_encoded_test = ohe.transform(test_cat_df[meta_categorical_cols])
X_num_test = test_quant_df[meta_numerical_cols].values
X_meta_test_final = np.hstack([X_cat_encoded_test, X_num_test])

X_mri_test = test_mri_df.drop(columns=['participant_id']).values
X_mri_test_pca = pca.transform(X_mri_test)

Saving Prediction

In [63]:
predictions = model.predict([X_meta_test_final, X_mri_test_pca])

y_adhd_pred = (predictions[0] > 0.5).astype(int)
y_sex_pred = (predictions[1] > 0.5).astype(int)

sample_submission['ADHD_Outcome'] = y_adhd_pred
sample_submission['Sex_F'] = y_sex_pred
sample_submission.to_csv("neural_network_predictions.csv", index=False)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


SUBMISSION ACCURACY: 74%

## LightGBM Model

In [59]:
X_combined = np.hstack([X_meta, X_mri_pca])
X_test_combined = np.hstack([X_meta_test_final, X_mri_test_pca])

In [60]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score
import lightgbm as lgb

# ADHD
X_train_adhd, X_val_adhd, y_train_adhd, y_val_adhd = train_test_split(
    X_combined, y_adhd, test_size=0.2, random_state=42, stratify=y_adhd
)

# Class weights
class_weights_adhd = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_adhd), y=y_train_adhd)
adhd_weight_dict = dict(zip(np.unique(y_train_adhd), class_weights_adhd))
adhd_sample_weights = np.array([adhd_weight_dict[y] for y in y_train_adhd])

# Train ADHD LightGBM
lgb_clf_adhd = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight=adhd_weight_dict, random_state=42)
lgb_clf_adhd.fit(X_train_adhd, y_train_adhd, sample_weight=adhd_sample_weights)

# Validation
y_val_pred_adhd = lgb_clf_adhd.predict(X_val_adhd)
print("ADHD Validation Accuracy:", accuracy_score(y_val_adhd, y_val_pred_adhd))


[LightGBM] [Info] Number of positive: 665, number of negative: 305
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023221 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 232519
[LightGBM] [Info] Number of data points in the train set: 970, number of used features: 976
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.314433 -> initscore=-0.779475
[LightGBM] [Info] Start training from score -0.779475
ADHD Validation Accuracy: 0.7860082304526749


In [61]:
# Sex
X_train_sex, X_val_sex, y_train_sex, y_val_sex = train_test_split(
    X_combined, y_sex, test_size=0.2, random_state=42, stratify=y_sex
)

# Class weights
class_weights_sex = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_sex), y=y_train_sex)
sex_weight_dict = dict(zip(np.unique(y_train_sex), class_weights_sex))
sex_sample_weights = np.array([sex_weight_dict[y] for y in y_train_sex])

# Train Sex LightGBM
lgb_clf_sex = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight=sex_weight_dict, random_state=42)
lgb_clf_sex.fit(X_train_sex, y_train_sex, sample_weight=sex_sample_weights)

# Validation
y_val_pred_sex = lgb_clf_sex.predict(X_val_sex)
print("Sex Validation Accuracy:", accuracy_score(y_val_sex, y_val_pred_sex))


[LightGBM] [Info] Number of positive: 333, number of negative: 637
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014033 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 232511
[LightGBM] [Info] Number of data points in the train set: 970, number of used features: 974
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.656701 -> initscore=0.648627
[LightGBM] [Info] Start training from score 0.648627
Sex Validation Accuracy: 0.6707818930041153


In [64]:
# Predict both targets
adhd_preds = lgb_clf_adhd.predict(X_test_combined)
sex_preds = lgb_clf_sex.predict(X_test_combined)

# Load submission format and assign predictions
sample_submission['ADHD_Outcome'] = adhd_preds
sample_submission['Sex_F'] = sex_preds

# Save final submission
sample_submission.to_csv("lightgbm_submission.csv", index=False)
print("LightGBM ADHD + Sex predictions saved to 'lightgbm_submission.csv'")


LightGBM ADHD + Sex predictions saved to 'lightgbm_submission.csv'


SUBMISSION ACCURACY: 76%